# 04 - Semantic Search Mini-Project

Semantic search retrieves documents by meaning rather than exact keyword overlap. It is one of the most common building blocks in LLM applications because it gives the model relevant context before generation.


## Concept Context

Semantic search is an application pattern, not only a similarity formula. A useful search system needs source documents, indexing rules, query handling, top-k ranking, evaluation labels, thresholds, and fallback behavior. Each of those choices affects whether an LLM receives useful context or misleading context.

This mini-project treats retrieval as a workflow that can be measured and debugged. The goal is to inspect evidence before trusting generated answers built on top of retrieved documents.

## Objectives

- Build a local search index over course lessons.
- Retrieve top-k candidates for support questions.
- Evaluate whether the expected topic appears in the top results.
- Add a confidence threshold and fallback behavior.


```mermaid
flowchart TD
    Q[User query] --> QV[Embed query]
    D[Document collection] --> DV[Embed documents]
    QV --> SIM[Compute similarities]
    DV --> SIM
    SIM --> RANK[Rank documents]
    RANK --> TOPK[Return top-k results]
    TOPK --> DECIDE[Use or fallback]
```


## What Makes This A Mini-Project

This notebook is intentionally more system-shaped than the previous ones. You are not only computing a similarity score; you are building a small retrieval workflow with documents, metadata, support questions, top-k results, manual evaluation, and fallback behavior.

That mirrors real RAG systems. Even if the production system uses a vector database and a provider embedding model, the core engineering loop is the same: index trusted documents, retrieve candidates, inspect quality, and decide what to do when the match is weak.


In [ ]:
# Search helpers are local and transparent so retrieval behavior can be debugged.
# Production systems may use embedding APIs and vector databases, but the evaluation loop is similar.
import csv
import math
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

TOKEN_PATTERN = re.compile(
    r"""
    [A-Za-z]+(?:'[A-Za-z]+)?  # words and contractions
    |\d+(?:\.\d+)?           # integers and decimals
    |[^\w\s]                 # punctuation and symbols
    """,
    re.VERBOSE,
)


@dataclass(frozen=True)
class TokenizationResult:
    text: str
    tokens: list[str]

    @property
    def count(self) -> int:
        return len(self.tokens)


def simple_tokenize(text: str) -> list[str]:
    """Split text into approximate tokens for transparent local practice."""
    return TOKEN_PATTERN.findall(text)


def inspect_text(text: str) -> TokenizationResult:
    return TokenizationResult(text=text, tokens=simple_tokenize(text))


def count_tokens(text: str) -> int:
    return len(simple_tokenize(text))


def token_budget_remaining(
    context_window: int, prompt_tokens: int, reserved_output_tokens: int
) -> int:
    return context_window - prompt_tokens - reserved_output_tokens


SparseVector = dict[str, float]

STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "as",
    "be",
    "by",
    "can",
    "do",
    "does",
    "for",
    "from",
    "how",
    "i",
    "in",
    "into",
    "is",
    "it",
    "me",
    "my",
    "of",
    "or",
    "should",
    "the",
    "this",
    "to",
    "what",
    "when",
    "whether",
    "why",
    "with",
}


@dataclass(frozen=True)
class EmbeddedCorpus:
    records: list[dict[str, str]]
    text_field: str
    idf: dict[str, float]
    vectors: list[SparseVector]


def content_tokens(text: str) -> list[str]:
    tokens = [token.lower() for token in simple_tokenize(text)]
    return [
        token
        for token in tokens
        if token not in STOPWORDS and any(char.isalnum() for char in token)
    ]


def combine_text_fields(record: dict[str, str], fields: list[str]) -> str:
    missing = [field for field in fields if field not in record]
    if missing:
        raise ValueError(f"Missing expected fields: {missing}")
    return " ".join(record.get(field, "") for field in fields)


def fit_idf(texts: list[str]) -> dict[str, float]:
    document_count = len(texts)
    document_frequency: Counter[str] = Counter()
    for text in texts:
        document_frequency.update(set(content_tokens(text)))
    return {
        token: math.log((1 + document_count) / (1 + frequency)) + 1
        for token, frequency in document_frequency.items()
    }


def vectorize(text: str, idf: dict[str, float]) -> SparseVector:
    tokens = content_tokens(text)
    if not tokens:
        return {}
    counts = Counter(tokens)
    total = len(tokens)
    vector = {
        token: (count / total) * idf.get(token, 0.0)
        for token, count in counts.items()
        if token in idf
    }
    norm = math.sqrt(sum(value * value for value in vector.values()))
    if norm == 0:
        return vector
    return {token: value / norm for token, value in vector.items()}


def embed_corpus(records: list[dict[str, str]], text_field: str) -> EmbeddedCorpus:
    if not records:
        raise ValueError("Cannot embed an empty corpus.")
    texts = [record[text_field] for record in records]
    idf = fit_idf(texts)
    vectors = [vectorize(text, idf) for text in texts]
    return EmbeddedCorpus(
        records=records, text_field=text_field, idf=idf, vectors=vectors
    )


def cosine_similarity(left: SparseVector, right: SparseVector) -> float:
    if not left or not right:
        return 0.0
    shared = set(left).intersection(right)
    return sum(left[token] * right[token] for token in shared)


def similarity_scores(corpus: EmbeddedCorpus, query: str) -> list[float]:
    query_vector = vectorize(query, corpus.idf)
    return [cosine_similarity(query_vector, vector) for vector in corpus.vectors]


@dataclass(frozen=True)
class SearchResult:
    rank: int
    score: float
    lesson_id: str
    title: str
    module: str
    description: str


class SemanticSearchIndex:
    def __init__(self, corpus: EmbeddedCorpus):
        self.corpus = corpus

    @classmethod
    def from_csv(
        cls, path: str | Path, text_fields: list[str] | None = None
    ) -> "SemanticSearchIndex":
        with Path(path).open(newline="", encoding="utf-8") as file:
            records = list(csv.DictReader(file))
        fields = text_fields or ["title", "module", "description"]
        for record in records:
            record["search_text"] = combine_text_fields(record, fields)
        return cls(embed_corpus(records, text_field="search_text"))

    def search(self, query: str, k: int = 3) -> list[SearchResult]:
        scores = similarity_scores(self.corpus, query)
        ranked_indices = sorted(
            range(len(scores)), key=lambda index: scores[index], reverse=True
        )[:k]
        results = []
        for rank, index in enumerate(ranked_indices, start=1):
            row = self.corpus.records[index]
            results.append(
                SearchResult(
                    rank=rank,
                    score=float(scores[index]),
                    lesson_id=row["lesson_id"],
                    title=row["title"],
                    module=row["module"],
                    description=row["description"],
                )
            )
        return results


def relevant_in_top_k(
    results: list[SearchResult], expected_topic: str, k: int = 3
) -> bool:
    expected = expected_topic.lower().strip().replace("_", " ")
    for result in results[:k]:
        searchable = (
            " ".join(
                [result.lesson_id, result.title, result.module, result.description]
            )
            .lower()
            .replace("_", " ")
        )
        if expected in searchable:
            return True
    return False

## Build The Search Index

The index embeds the course catalog and returns ranked lesson candidates for each support question.


In [ ]:
# Load local CSV data; keeping data on disk makes the examples reproducible.
catalog_path = Path("data") / "course_catalog.csv"
questions_path = Path("data") / "support_questions.csv"
index = SemanticSearchIndex.from_csv(catalog_path)
with questions_path.open(newline="", encoding="utf-8") as file:
    questions = list(csv.DictReader(file))
print(f"Loaded {len(questions)} support questions")

## Inspect Top-k Results

Top-3 often matters more than top-1 because a downstream RAG prompt may include several chunks. Inspect the candidates before trusting the system.


## Retrieval Evaluation Loop

```mermaid
flowchart LR
    A[Collect realistic questions] --> B[Label expected source or topic]
    B --> C[Run top-k retrieval]
    C --> D[Inspect ranked candidates]
    D --> E[Record recall@k and failure notes]
    E --> F[Improve text, chunking, metadata, or threshold]
    F --> C
```

The loop is more important than any single score. Retrieval quality usually improves through repeated inspection of failure cases.


In [ ]:
# Inspect one labeled support question before running the full evaluation loop.
question = questions[2]
results = index.search(question["question"], k=3)
print(f"Question: {question['question']}")
print(f"Expected topic: {question['expected_topic']}\n")
for result in results:
    print(
        f"{result.rank}. {result.title} | score={result.score:.3f} | module={result.module}"
    )

## Manual Evaluation

This check asks whether the expected topic appears somewhere in the top-k result metadata. In a real project, use explicit human relevance labels.


In [ ]:
# Evaluate every labeled question so retrieval quality is measured, not guessed.
evaluation_rows = []
for row in questions:
    results = index.search(row["question"], k=3)
    evaluation_rows.append(
        {
            "question_id": row["question_id"],
            "expected_topic": row["expected_topic"],
            "top_1": results[0].title,
            "top_1_score": round(results[0].score, 3),
            "relevant_in_top_3": relevant_in_top_k(results, row["expected_topic"], k=3),
        }
    )
for row in evaluation_rows:
    print(row)
print(
    "Recall@3:",
    sum(row["relevant_in_top_3"] for row in evaluation_rows),
    "/",
    len(evaluation_rows),
)

## Add A Confidence Threshold

Search always returns something unless the application has a fallback rule. Thresholds should be calibrated with realistic examples.


In [ ]:
# Apply a simple score threshold so weak nearest neighbors can fall back.
threshold = 0.15
queries = [
    "How do I estimate token cost?",
    "How do I search documents by meaning?",
    "How do I repair a bicycle chain?",
]
for query in queries:
    results = index.search(query, k=3)
    top = results[0]
    decision = "fallback" if top.score < threshold else "use retrieved candidates"
    print(f"{query}")
    print(f"  top result: {top.title} ({top.score:.3f})")
    print(f"  decision: {decision}")

## How To Read The Evaluation Rows

Each printed row has four useful pieces of evidence: the question ID, expected topic, top-1 result, and whether the expected topic appears in the top three. If top-1 is wrong but top-3 contains the right topic, a downstream RAG prompt may still succeed if it receives multiple chunks. If the relevant topic is absent from top-3, the retrieval step needs work.

Manual labels are small here, but the habit scales. In production, a labeled evaluation set is one of the fastest ways to detect retrieval regressions.


## Product Design Questions

Before connecting semantic search to an LLM, answer:

- What source documents are allowed?
- How often are documents updated?
- Should results include citations?
- What happens when no result is relevant?
- Should the model answer without retrieved context?


## Product Design Reference Answers

- Allowed sources should be trusted, versioned course or product documents, not arbitrary web pages.
- Update frequency should match business risk: high-change policies need frequent refresh and re-indexing; stable conceptual material can refresh less often.
- Results should include citations or source IDs so generated answers can be audited.
- When no result is relevant, the system should ask for clarification, suggest nearby topics, or route to another support path.
- The model should not answer without retrieved context when the task requires grounded, auditable information.

## What To Inspect

Look for surprising top-1 results, relevant documents that appear only in top-3, and unrelated questions that still receive a confident-looking result.


## Practice

Edit `your_questions` with three realistic questions: one about tokenization, one about embeddings, and one unrelated question.


## Thresholds Are Product Policy

A threshold is not only a math setting. It controls the user experience. A low threshold may answer more questions but risks unsupported answers. A high threshold may be safer but can frustrate users by escalating too often.

Good fallback behavior should be specific. Instead of returning a vague error, the system can ask a clarifying question, suggest related topics, or route to a human support path.


In [ ]:
# Editable practice queries for checking in-scope and out-of-scope behavior.
your_questions = [
    "How can I know if my prompt is too long?",
    "What does cosine similarity compare?",
    "What should I cook tonight?",
]
for query in your_questions:
    results = index.search(query, k=3)
    print(f"\n{query}")
    for result in results:
        print(f"  {result.rank}. {result.title} | score={result.score:.3f}")

## Practice Reference Answer

Expected behavior: the tokenization question should retrieve a tokenization or cost module, the embedding question should retrieve an embeddings or semantic-search module, and the unrelated question should either score weakly or be flagged by the threshold. If the unrelated question receives a plausible title, that is not necessarily a bug; it is evidence that thresholding and fallback policy are required.

## Checkpoint

Answer the questions, then compare with the reference answers below:

1. Why does semantic search use top-k instead of only top-1?
2. What does recall@k measure?
3. Why is a fallback threshold useful?
4. What metadata would help debug a real search result?


## Checkpoint Reference Answers

1. Top-k is useful because the best answer may not be ranked first, and downstream RAG prompts often include several candidate chunks.
2. Recall@k measures whether a relevant source or topic appears anywhere in the first k retrieved results.
3. A fallback threshold prevents the application from treating every nearest neighbor as a valid answer source.
4. Useful debug metadata includes source ID, title, module, score, chunk text, timestamp, query text, and the threshold decision.